# CUDA Kernels — PMPP Chapters 3–5

Make sure you have a GPU runtime: **Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install nvcc4jupyter -q
%load_ext nvcc4jupyter

## 1. Hello CUDA — one printf per thread

In [ ]:
%%cuda
#include <stdio.h>

__global__ void hello_kernel() {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from block %d, thread %d (global id: %d)\n",
           blockIdx.x, threadIdx.x, tid);
}

int main() {
    // 2 blocks, 4 threads each = 8 total threads
    hello_kernel<<<2, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

## 2. Vector Addition (Ch. 3)

Each thread computes one element: `c[i] = a[i] + b[i]`

The full CUDA program structure from Ch. 3:
- `cudaMalloc` — allocate device memory
- `cudaMemcpy` — transfer host → device
- kernel launch
- `cudaMemcpy` — transfer device → host
- `cudaFree`

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>

__global__ void vector_add(float *a, float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] + b[i];
}

int main() {
    int N = 1024;
    size_t size = N * sizeof(float);

    // Allocate host memory
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);

    for (int i = 0; i < N; i++) { h_a[i] = i; h_b[i] = i * 2; }

    // Allocate device memory (cudaMalloc)
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // Copy host -> device (cudaMemcpy)
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int BLOCK = 256;
    int GRID  = (N + BLOCK - 1) / BLOCK;
    vector_add<<<GRID, BLOCK>>>(d_a, d_b, d_c, N);

    // Copy device -> host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Verify
    int ok = 1;
    for (int i = 0; i < N; i++)
        if (h_c[i] != h_a[i] + h_b[i]) { ok = 0; break; }
    printf("%s\n", ok ? "Vector addition correct!" : "WRONG");
    printf("c[0]=%.0f  c[1]=%.0f  c[1023]=%.0f\n", h_c[0], h_c[1], h_c[1023]);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

## 3. Naive Matrix Multiplication (Ch. 3)

Each thread computes one output element by reading a full row of M and column of N from **global memory**.

For a 1000×1000 matrix: **2 billion global memory reads** (Ch. 3 exercise answer).

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH 256

__global__ void matmul_naive(float *M, float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < width && col < width) {
        float sum = 0.0f;
        for (int k = 0; k < width; k++)
            sum += M[row * width + k] * N[k * width + col]; // global mem reads
        P[row * width + col] = sum;
    }
}

int main() {
    int n = WIDTH * WIDTH;
    size_t size = n * sizeof(float);

    float *h_M = (float*)malloc(size);
    float *h_N = (float*)malloc(size);
    float *h_P = (float*)malloc(size);

    // Identity matrix * identity matrix = identity
    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
            h_N[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size);
    cudaMalloc(&d_N, size);
    cudaMalloc(&d_P, size);

    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(16, 16);
    dim3 grid(WIDTH / 16, WIDTH / 16);
    matmul_naive<<<grid, block>>>(d_M, d_N, d_P, WIDTH);

    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    // Verify: result should be identity
    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++) {
            float expected = (i == j) ? 1.0f : 0.0f;
            if (fabsf(h_P[i * WIDTH + j] - expected) > 1e-3f) ok = 0;
        }
    printf("%s\n", ok ? "Naive matmul correct!" : "WRONG");
    printf("P[0][0]=%.1f  P[0][1]=%.1f  P[1][1]=%.1f\n",
           h_P[0], h_P[1], h_P[WIDTH + 1]);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
    return 0;
}

## 4. Tiled Matrix Multiplication with Shared Memory (Ch. 4–5)

Each block loads a **tile** of M and N into `__shared__` memory (fast on-chip SRAM), then all threads in the block reuse it — reducing global memory reads by `TILE_WIDTH`x.

- `__shared__` — shared memory, lives on-chip, shared within a block (Ch. 5)
- `__syncthreads()` — barrier so all threads finish loading before computing (Ch. 4.3)

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH      256
#define TILE_WIDTH 16

__global__ void matmul_tiled(float *M, float *N, float *P, int width) {
    __shared__ float Mds[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Nds[TILE_WIDTH][TILE_WIDTH];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE_WIDTH + ty;
    int col = blockIdx.x * TILE_WIDTH + tx;

    float sum = 0.0f;

    for (int t = 0; t < width / TILE_WIDTH; t++) {
        // All threads in block collaboratively load one tile into shared mem
        Mds[ty][tx] = M[row * width + (t * TILE_WIDTH + tx)];
        Nds[ty][tx] = N[(t * TILE_WIDTH + ty) * width + col];

        __syncthreads(); // wait for all threads to finish loading

        for (int k = 0; k < TILE_WIDTH; k++)
            sum += Mds[ty][k] * Nds[k][tx]; // reads from fast shared memory

        __syncthreads(); // wait before overwriting tile in next iteration
    }

    if (row < width && col < width)
        P[row * width + col] = sum;
}

int main() {
    int n = WIDTH * WIDTH;
    size_t size = n * sizeof(float);

    float *h_M = (float*)malloc(size);
    float *h_N = (float*)malloc(size);
    float *h_P = (float*)malloc(size);

    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
            h_N[i * WIDTH + j] = (i == j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size);
    cudaMalloc(&d_N, size);
    cudaMalloc(&d_P, size);

    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(TILE_WIDTH, TILE_WIDTH);
    dim3 grid(WIDTH / TILE_WIDTH, WIDTH / TILE_WIDTH);
    matmul_tiled<<<grid, block>>>(d_M, d_N, d_P, WIDTH);

    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++) {
            float expected = (i == j) ? 1.0f : 0.0f;
            if (fabsf(h_P[i * WIDTH + j] - expected) > 1e-3f) ok = 0;
        }
    printf("%s\n", ok ? "Tiled matmul correct!" : "WRONG");
    printf("Global memory reads reduced by %dx vs naive\n", TILE_WIDTH);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
    return 0;
}